# Nền tảng 6 — Bên trong Transformer của nanochat

Bạn đã nắm sơ đồ khối Transformer và công thức attention. Notebook này lấp phần còn lại: những thành phần mà model
của project **thực sự** dùng nhưng không có trong bài giảng Transformer chuẩn — RoPE, GQA, KV cache, cửa sổ trượt,
value embedding, RMSNorm, ReLU², và optimizer Muon.

Cách tiếp cận: mỗi thành phần được cài lại bằng torch thuần trong vài dòng, chạy trên CPU với tensor nhỏ, rồi
kiểm chứng bằng một phép so sánh số. Cuối cùng ghép tất cả thành một model nhỏ chạy được, và kiểm tra đúng tính
chất mà `vitok/eval.py` dựa vào để chấm điểm hợp lệ.

**Chạy bằng kernel pixi của project.** Không cần GPU.

In [ ]:
import json
import math
import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "third_party" / "nanochat"))

from nanochat.gpt import apply_rotary_emb, has_ve

torch.manual_seed(0)
print("torch", torch.__version__, "| thiết bị: CPU")

## 1. Attention, và chỗ trống mà nó để lại

Công thức bạn đã biết:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_h}}\right)V$$

Chia cho $\sqrt{d_h}$ vì tích vô hướng của hai vector $d_h$ chiều có phương sai tỷ lệ $d_h$; không chia thì
softmax bão hoà và gradient biến mất.

Chỗ trống: công thức này **hoàn toàn không biết thứ tự**. Hoán vị các token đầu vào thì đầu ra hoán vị theo,
không có gì thay đổi. Cell dưới kiểm chứng điều đó, và nó là lý do phải thêm thông tin vị trí.

In [ ]:
B, T, dh = 1, 6, 16
q = torch.randn(B, T, dh)
k, v = torch.randn(B, T, dh), torch.randn(B, T, dh)


def attention(q, k, v, mask=None):
    diem = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        diem = diem.masked_fill(mask, float("-inf"))
    return torch.softmax(diem, dim=-1) @ v


ra = attention(q, k, v)
hoan_vi = torch.randperm(T)
ra_hv = attention(q[:, hoan_vi], k[:, hoan_vi], v[:, hoan_vi])
print("attention không mask, hoán vị đầu vào rồi hoán vị lại đầu ra:")
print("  sai khác lớn nhất:", (ra_hv - ra[:, hoan_vi]).abs().max().item())
print("  => attention là hàm ĐỐI XỨNG theo thứ tự: không có thông tin vị trí.")

## 2. RoPE: nhúng vị trí bằng phép quay

Cách cũ (GPT-2) là cộng một vector vị trí học được vào embedding. RoPE (rotary position embedding) làm khác: nó
**quay** $q$ và $k$ một góc tỷ lệ với vị trí.

Chia vector $d_h$ chiều thành $d_h/2$ cặp. Cặp thứ $i$ được quay góc $t\theta_i$ tại vị trí $t$, với

$$\theta_i = \mathrm{base}^{-2i/d_h}, \qquad \mathrm{base} = 100\,000 \text{ trong nanochat}$$

**Ký hiệu mới:** $\theta_i$ — tốc độ quay (radian mỗi vị trí) của cặp chiều thứ $i$; cặp $i = 0$ quay nhanh nhất.

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos t\theta_i & -\sin t\theta_i \\ \sin t\theta_i & \cos t\theta_i \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

**Ký hiệu mới:** $q'$ — vector sau khi quay; ma trận $2 \times 2$ là phép quay góc $t\theta_i$, ký hiệu gọn $R_t$.

Tính chất làm nó hoạt động: phép quay bảo toàn tích vô hướng, và tích vô hướng của hai vector đã quay **chỉ phụ
thuộc hiệu vị trí**:

$$\langle R_m q,\ R_n k\rangle = \langle R_{m-n} q,\ k \rangle$$

**Ký hiệu mới:** $\langle u, v \rangle$ — tích vô hướng (một phần tử của $QK^\top$); $m$, $n$ — vị trí của query
và của key.

**Vì sao:** $R_n^\top = R_{-n}$ và quay liên tiếp thì cộng góc, nên $R_n^\top R_m = R_{m-n}$.

Nghĩa là điểm attention giữa vị trí $m$ và $n$ phụ thuộc $m - n$, tức **vị trí tương đối**, dù ta chỉ mã hoá vị
trí tuyệt đối. Đó là điều mà cách cộng vector vị trí không làm được.

Chi tiết cài đặt trong nanochat: nó tách nửa đầu/nửa sau vector thay vì các cặp liền kề, và quay theo chiều
$-\theta$ (ghi chú trong `gpt.py` nói rõ đây là chuyển vị của quy ước sách vở). Cả hai đều không đổi tính chất
trên, vì chỉ có phép quay **tương đối** giữa $q$ và $k$ mới đi vào kết quả.

In [ ]:
def rope_cos_sin(T, dh, base=100_000):
    kenh = torch.arange(0, dh, 2, dtype=torch.float32)
    inv_freq = 1.0 / (base ** (kenh / dh))                 # theta_i
    t = torch.arange(T, dtype=torch.float32)
    goc = torch.outer(t, inv_freq)                          # (T, dh/2)
    return goc.cos()[None, :, None, :], goc.sin()[None, :, None, :]


T, dh = 32, 64
cos, sin = rope_cos_sin(T, dh)
q = torch.randn(1, T, 1, dh)
k = torch.randn(1, T, 1, dh)
q_r, k_r = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)

print("1) phép quay bảo toàn độ dài:")
print(f"   |q| trước {q[0, 5, 0].norm():.6f} | sau {q_r[0, 5, 0].norm():.6f}")

print("\n2) tích vô hướng chỉ phụ thuộc hiệu vị trí (cùng một cặp q,k đặt ở các vị trí khác nhau):")
q_hang = q[:, :1].expand(1, T, 1, dh).contiguous()          # cùng một vector q ở mọi vị trí
k_hang = k[:, :1].expand(1, T, 1, dh).contiguous()
qr = apply_rotary_emb(q_hang, cos, sin)
kr = apply_rotary_emb(k_hang, cos, sin)
for m, n in ((10, 4), (20, 14), (31, 25)):
    print(f"   m={m:2d}, n={n:2d}, m-n={m - n}: <q_m, k_n> = {(qr[0, m, 0] @ kr[0, n, 0]).item():+.6f}")
print("   => ba giá trị bằng nhau vì cùng hiệu vị trí 6.")

print("\n3) bước sóng của từng cặp chiều (2π/θ_i): chiều đầu quay nhanh, chiều cuối gần như đứng yên")
inv = 1.0 / (100_000 ** (torch.arange(0, dh, 2).float() / dh))
for i in (0, 8, 16, 31):
    print(f"   cặp {i:2d}: θ = {inv[i]:.2e} -> bước sóng {2 * math.pi / inv[i]:12,.0f} vị trí")

Vì sao `base` lớn (100.000 thay vì 10.000): bước sóng của cặp chậm nhất tỷ lệ với base, nên base lớn cho phép
model phân biệt vị trí trong ngữ cảnh dài hơn mà không bị "quấn vòng". Đây cũng là nút vặn chính khi người ta
kéo dài ngữ cảnh của một model đã huấn luyện xong.

## 3. Nhiều đầu và GQA

**Multi-head**: chia $d$ chiều residual thành $h$ đầu, mỗi đầu $d_h = d/h$ chiều, chạy attention độc lập rồi nối
lại. Mỗi đầu học một kiểu quan hệ khác nhau.

**GQA** (grouped-query attention) giảm số đầu của $K$ và $V$ xuống $h_{kv} < h$, mỗi nhóm $h/h_{kv}$ đầu query
dùng chung một cặp K/V. Lý do không phải tiết kiệm tham số mà là tiết kiệm **KV cache** lúc sinh văn bản (mục 4),
vì cache tỷ lệ với $h_{kv}$.

Cấu hình của project: `head_dim = 128` và `n_kv_head = n_head`, tức **không** dùng GQA thu gọn. Với d8 thì
$h = 512/128 = 4$ đầu.

In [ ]:
def attention_gqa(q, k, v, n_head, n_kv_head, causal=True):
    """q: (B,T,n_head,dh); k,v: (B,T,n_kv_head,dh)"""
    B, T, _, dh = q.shape
    lap = n_head // n_kv_head
    k = k.repeat_interleave(lap, dim=2)                      # mỗi nhóm query dùng chung một K/V
    v = v.repeat_interleave(lap, dim=2)
    q, k, v = (x.transpose(1, 2) for x in (q, k, v))         # -> (B, head, T, dh)
    ra = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    return ra.transpose(1, 2).reshape(B, T, n_head * dh)


B, T, dh = 2, 16, 32
for n_head, n_kv in ((4, 4), (4, 2), (4, 1)):
    q = torch.randn(B, T, n_head, dh)
    k, v = torch.randn(B, T, n_kv, dh), torch.randn(B, T, n_kv, dh)
    ra = attention_gqa(q, k, v, n_head, n_kv)
    cache_bytes = 2 * n_kv * dh * 2                           # K và V, fp16, mỗi token mỗi lớp
    print(f"  h={n_head}, h_kv={n_kv}: đầu ra {tuple(ra.shape)} | KV cache {cache_bytes} byte/token/lớp")

## 4. KV cache: vì sao sinh văn bản không phải chạy lại cả câu

Lúc **huấn luyện**, cả chuỗi $T$ token được xử lý một lần, song song. Lúc **sinh**, token ra từng cái một, và
token mới cần attention tới toàn bộ quá khứ.

> **Ký hiệu $O(\cdot)$ (big-O):** chi phí tăng theo cỡ đầu vào ra sao, bỏ qua hằng số. $O(n)$: gấp đôi $n$ thì gấp
> đôi chi phí. $O(n^2)$: gấp đôi $n$ thì gấp bốn.

Không có cache, sinh $T$ token tốn $\sum_{t=1}^{T} O(t) = O(T^2)$ lần tính lại **cùng những $K, V$ cũ** — chúng
không đổi, vì attention là nhân quả: $K_j, V_j$ chỉ phụ thuộc token $j$ và các token trước nó.

**KV cache** lưu lại $K, V$ của mọi vị trí đã sinh. Mỗi bước mới chỉ tính $K, V$ cho **một** token rồi nối vào
cache. Chi phí mỗi bước từ $O(t \cdot d^2)$ xuống $O(d^2)$ cộng phần attention $O(t \cdot d)$.

Cái giá là bộ nhớ:

$$\text{bộ nhớ cache} = 2 \times L \times T \times h_{kv} \times d_h \times \text{(số byte mỗi phần tử)}$$

(hệ số 2 cho K và V). Cell dưới cài cả hai cách rồi đo thời gian thật trên CPU.

In [ ]:
def sinh_khong_cache(x_emb, W_k, W_v, W_q, n_buoc):
    """Mỗi bước tính lại K,V cho TOÀN BỘ chuỗi."""
    chuoi = x_emb
    for _ in range(n_buoc):
        q, k, v = chuoi @ W_q, chuoi @ W_k, chuoi @ W_v
        ra = attention(q[None], k[None], v[None],
                       mask=torch.triu(torch.ones(len(chuoi), len(chuoi), dtype=torch.bool), 1))[0]
        chuoi = torch.cat([chuoi, ra[-1:]], dim=0)
    return chuoi


def sinh_co_cache(x_emb, W_k, W_v, W_q, n_buoc):
    """K,V của quá khứ được giữ lại; mỗi bước chỉ tính cho token mới."""
    K, V = x_emb @ W_k, x_emb @ W_v
    chuoi = x_emb
    for _ in range(n_buoc):
        x_moi = chuoi[-1:]
        q = x_moi @ W_q
        K = torch.cat([K, x_moi @ W_k], dim=0)               # <- nối vào cache
        V = torch.cat([V, x_moi @ W_v], dim=0)
        ra = attention(q[None], K[None], V[None])[0]
        chuoi = torch.cat([chuoi, ra[-1:]], dim=0)
    return chuoi


d = 256
W_q, W_k, W_v = (torch.randn(d, d) / math.sqrt(d) for _ in range(3))
x0 = torch.randn(32, d)

for n_buoc in (50, 200):
    t0 = time.time()
    sinh_khong_cache(x0, W_k, W_v, W_q, n_buoc)
    t_khong = time.time() - t0
    t0 = time.time()
    sinh_co_cache(x0, W_k, W_v, W_q, n_buoc)
    t_co = time.time() - t0
    print(f"  sinh {n_buoc:3d} token: không cache {t_khong:.3f}s | có cache {t_co:.3f}s | nhanh hơn {t_khong / t_co:.1f}×")

In [ ]:
print("bộ nhớ KV cache của model project (fp16, 2 byte mỗi phần tử):\n")
print(" model | lớp | h_kv | d_h | byte/token | cache cho 1.024 token | cho 8.192 token")
for depth in (6, 8, 10):
    d_model = 64 * depth
    h_kv = d_model // 128
    moi_token = 2 * depth * h_kv * 128 * 2
    print(f" d{depth:<4} | {depth:3d} | {h_kv:4d} | 128 | {moi_token:10,} | {moi_token * 1024 / 1e6:19.1f} MB |"
          f" {moi_token * 8192 / 1e6:13.1f} MB")

print("\nĐể so: nếu dùng GQA với h_kv = 1 thì cache của d10 giảm đúng 5 lần.")
print("Đây là lý do mọi model phục vụ thật đều dùng GQA, dù project không cần vì không sinh văn bản quy mô lớn.")

## 5. Cửa sổ trượt và vì sao project tắt nó

Sliding window attention giới hạn mỗi token chỉ nhìn $w$ token gần nhất thay vì toàn bộ quá khứ. Chi phí
attention đổi từ $O(T^2)$ sang $O(Tw)$, và KV cache chỉ cần giữ $w$ vị trí.

nanochat cấu hình bằng chuỗi `window_pattern`: `"SSSL"` nghĩa là ba lớp cửa sổ ngắn ($w = T/4$) rồi một lớp toàn
bộ ngữ cảnh, lặp lại; lớp cuối **luôn** là L. Ý tưởng: lớp ngắn lo quan hệ cục bộ, lớp dài lo quan hệ xa.

Project đặt `--window-pattern L` (mọi lớp đều nhìn toàn bộ). Lý do nằm ở đường mã chạy trên T4. FlashAttention 3
chỉ có trên GPU Hopper; trên T4 (Turing) nanochat rơi về hàm `_sdpa_attention` trong `nanochat/flash_attention.py`.
Hàm đó **vẫn hỗ trợ** cửa sổ trượt, nhưng bằng cách dựng một mặt nạ boolean $T \times T$ tường minh rồi truyền vào
`F.scaled_dot_product_attention(..., attn_mask=mask)`. Hệ quả:

- vẫn tính đủ $T \times T$ điểm số rồi mới che, nên **không tiết kiệm FLOPs** như FA3;
- mất đường nhanh `is_causal=True` mà PyTorch dành cho mặt nạ nhân quả thuần, và tốn thêm bộ nhớ cho mặt nạ.

Tức trên T4, cửa sổ trượt chỉ đổi kiến trúc mà không đem lại lợi ích tốc độ nào. Với `L`, mọi lớp đi đường
`is_causal=True` nhanh nhất và mọi điều kiện chạy đúng cùng một đường mã.

(Ghi chú: `CLAUDE.md` của project viết "SDPA has no sliding window". Đúng hơn là: *SDPA không có tham số cửa sổ
riêng; nanochat mô phỏng nó bằng mặt nạ, và việc đó không tiết kiệm gì trên T4*.)

Cell dưới cài mask cửa sổ trượt bằng tay và tính phần FLOPs **về lý thuyết** tiết kiệm được với một kernel biết
bỏ qua phần bị che (như FA3) — đúng phần mà T4 không lấy được.

In [ ]:
def mask_cua_so(T, w):
    i = torch.arange(T)[:, None]
    j = torch.arange(T)[None, :]
    return (j > i) | (j < i - w + 1)                          # True = bị chặn


T = 12
print("mask với cửa sổ w=4 (X = nhìn được):")
m = mask_cua_so(T, 4)
for r in range(T):
    print("   ", "".join("." if m[r, c] else "X" for c in range(T)))

print("\nphần attention của d8 với các cấu hình cửa sổ (T = 1024):")
h, dh, L, T_ctx = 4, 128, 8, 1024
for ten, pattern in (("L (project)", "L"), ("SSSL (mặc định nanochat)", "SSSL")):
    tong = 0
    for i in range(L):
        ch = pattern[i % len(pattern)] if i < L - 1 else "L"
        w = T_ctx if ch == "L" else -(-T_ctx // 4 // 128) * 128
        tong += 12 * h * dh * min(w, T_ctx)
    print(f"  {ten:26s}: {tong:,} FLOPs/token ({tong / (12 * h * dh * T_ctx * L):.0%} so với toàn bộ)")

## 6. Value embedding: đường tắt từ token tới giá trị

Đây là thành phần chiếm nhiều tham số nhất mà bài giảng Transformer chuẩn không có. Ý tưởng (ResFormer /
modded-nanogpt): ngoài embedding đầu vào, mỗi **lớp được chọn** có thêm một bảng embedding riêng, và giá trị tra
được cộng vào $V$ của lớp đó:

$$V^{(l)} = V^{(l)}_{\text{thường}} + g^{(l)}(x) \odot E^{(l)}[\text{token}]$$

**Ký hiệu mới**
- chỉ số trên $(l)$ — số thứ tự lớp, không phải số mũ
- $E^{(l)}[\text{token}]$ — hàng của bảng value embedding lớp $l$, ứng với token hiện tại
- $\odot$ — nhân từng phần tử

với cổng $g^{(l)}(x) = 3\,\sigma\big(W_g\, \tilde x_{[:12]}\big)$ — một cổng học được, nằm trong khoảng $(0, 3)$,
tính từ **12 chiều đầu** của residual đã chuẩn hoá $\tilde x$ (`ve_gate_channels = 12` trong `gpt.py`), và cho ra
**một giá trị cho mỗi đầu KV** — nên $W_g$ chỉ có $12 \times h_{kv}$ tham số.

Tác dụng: thông tin **token gốc** đi thẳng tới $V$ của lớp sâu mà không phải sống sót qua mọi lớp phía trước.
Residual stream vốn đã làm việc tương tự, nhưng nó bị mọi lớp viết đè lên; value embedding là một đường riêng,
không bị ghi đè.

Quy tắc chọn lớp, đọc thẳng từ `gpt.py`:

```python
def has_ve(layer_idx, n_layer):
    return layer_idx % 2 == (n_layer - 1) % 2
```

tức các lớp **cùng chẵn lẻ với lớp cuối**. Với $L$ chẵn, đó là các lớp lẻ và $n_{ve} = L/2$.

In [ ]:
for L in (6, 8, 10):
    lop = [i for i in range(L) if has_ve(i, L)]
    print(f"  L={L:2d}: value embedding ở các lớp {lop} -> {len(lop)} bảng")

d_model, V_pad = 640, 16064
n_ve = sum(1 for i in range(10) if has_ve(i, 10))
print(f"\nd10: {n_ve} bảng × {V_pad:,} × {d_model} = {n_ve * V_pad * d_model:,} tham số")
print(f"     = {n_ve * V_pad * d_model / 121_119_066:.1%} tổng tham số của model")

# cổng: 3·sigmoid, nên nằm trong (0, 3) và khởi tạo quanh 1.5
x = torch.randn(1000, 12)                      # 12 chiều đầu của residual đã chuẩn hoá
W_g = torch.randn(12, 4) * 0.02                # 4 = số đầu KV ở d8
cong = 3 * torch.sigmoid(x @ W_g)
print(f"\ncổng 3·sigmoid với trọng số khởi tạo nhỏ: khoảng giá trị [{cong.min():.3f}, {cong.max():.3f}],"
      f" trung bình {cong.mean():.3f}")
print("=> lúc bắt đầu, value embedding được cộng vào với hệ số ~1,5 và model tự học tăng/giảm.")

## 7. Các chi tiết còn lại của khối nanochat

| Thành phần | Công thức | Vì sao |
|---|---|---|
| **RMSNorm** | $\mathrm{RMS}(x) = x / \sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}$ | như LayerNorm nhưng bỏ phép trừ trung bình → rẻ hơn, thực nghiệm không kém; nanochat dùng `F.rms_norm` **không có** hệ số nhân học được, và chuẩn hoá cả embedding ngay sau khi tra |
| **Softcap logits** | $z \leftarrow 15\tanh(z/15)$ trước softmax | chặn logits trong $(-15, 15)$ để ổn định huấn luyện; áp cả khi đánh giá, nên nat mà `vitok/eval.py` đo là của phân phối sau softcap |
| **ReLU²** | $\mathrm{MLP}(x) = W_2\,\mathrm{relu}(W_1x)^2$ | thay GELU; đơn giản, thực nghiệm tốt trong modded-nanogpt |
| **Không bias** | mọi `Linear(bias=False)` | bias gần như không giúp khi đã có normalization |
| **Embedding không buộc** | `wte` và `lm_head` là hai ma trận riêng | cho phép learning rate khác nhau (0,3 vs 0,008) |
| **`resid_lambdas`, `x0_lambdas`** | $x \leftarrow \lambda_l x + \mu_l x_0$ | scalar học được mỗi lớp, trộn lại embedding gốc |
| **smear** | trộn embedding của token trước vào token hiện tại | thông tin kiểu bigram, rất rẻ |
| **backout** | trừ residual giữa tầng trước lớp norm cuối | bỏ bớt đặc trưng tầng thấp trước khi dự đoán |

Ba dòng cuối là các thủ thuật của modded-nanogpt; chúng chiếm rất ít tham số (vài chục, xem notebook 03) nhưng
đáng biết vì chúng giải thích tại sao đếm tham số bằng $12d^2L$ vẫn lệch một chút.

In [ ]:
def rmsnorm(x, eps=1e-6):
    return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)


def layernorm(x, eps=1e-6):
    x = x - x.mean(-1, keepdim=True)
    return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)


x = torch.randn(4, 512) * 3 + 1.0
print(f"đầu vào : trung bình {x.mean():+.3f}, rms {x.pow(2).mean().sqrt():.3f}")
print(f"RMSNorm : trung bình {rmsnorm(x).mean():+.3f}, rms {rmsnorm(x).pow(2).mean().sqrt():.3f}  (không trừ trung bình nên nó vẫn khác 0)")
print(f"LayerNorm: trung bình {layernorm(x).mean():+.3f}, rms {layernorm(x).pow(2).mean().sqrt():.3f}  (ép trung bình về 0)")
print(f"khớp nhau với torch: {torch.allclose(rmsnorm(x), F.rms_norm(x, (x.size(-1),)), atol=1e-5)}")

z = torch.linspace(-2, 2, 9)
print("\nReLU²(z) so với GELU(z):")
print("  z     ", " ".join(f"{t:+6.2f}" for t in z))
print("  relu² ", " ".join(f"{t:+6.2f}" for t in F.relu(z) ** 2))
print("  gelu  ", " ".join(f"{t:+6.2f}" for t in F.gelu(z)))

## 8. Muon: optimizer cho các ma trận

nanochat dùng **hai** optimizer cùng lúc:

- **AdamW** cho embedding, unembedding và các tham số 0/1 chiều.
- **Muon** cho mọi ma trận 2 chiều trong thân model.

Ý tưởng của Muon (MomentUm Orthogonalized by Newton-schulz): chạy SGD-momentum như thường, nhưng trước khi cập
nhật thì **trực giao hoá** ma trận cập nhật — thay $G$ bằng ma trận trực giao gần nó nhất. Nếu $G = U\Sigma V^\top$
là phân tích SVD thì ma trận đó là

$$\mathrm{ortho}(G) = UV^\top$$

**Ký hiệu mới**
- $G = U\Sigma V^\top$ — phân tích giá trị kỳ dị (SVD) của ma trận cập nhật $G$
- $U$, $V$ — các "hướng" (cột trực chuẩn); $V$ ở đây **không** phải value của attention
- $\Sigma$ — ma trận đường chéo chứa các giá trị kỳ dị: độ lớn của $G$ theo từng hướng

tức **mọi giá trị kỳ dị bị ép về 1**. Diễn giải: gradient của một ma trận thường bị chi phối bởi vài hướng có giá
trị kỳ dị lớn, nên bước cập nhật đi gần như chỉ theo vài hướng đó. Ép phổ về đều làm mọi hướng được cập nhật cùng
một "độ dài", nên mỗi bước khai thác được nhiều chiều hơn.

Tính SVD mỗi bước thì quá đắt, nên Muon dùng **lặp Newton–Schulz**: một đa thức ma trận $X \mapsto aX + bX(X^\top X)
+ cX(X^\top X)^2$ hội tụ về $UV^\top$ sau vài vòng, chỉ gồm phép nhân ma trận nên chạy được ở bfloat16 trên GPU.
nanochat dùng 5 vòng với bộ hệ số "Polar Express".

Cell dưới cài **phần lõi** của bước đó — chuẩn hoá rồi 5 vòng lặp đa thức với đúng bộ hệ số của nanochat — và in
phổ giá trị kỳ dị sau mỗi vòng. Bản thật trong `nanochat/optim.py` còn thêm ba bước không vẽ ở đây: cân bằng
chuẩn theo hàng trước khi trực giao hoá (MuonEq), đặt lại chuẩn Frobenius sau đó, và một bước giảm phương sai
kiểu Adam theo từng hàng/cột.

In [ ]:
polar_express_coeffs = [
    (8.156554524902461, -22.48329292557795, 15.878769915207462),
    (4.042929935166739, -2.808917465908714, 0.5000178451051316),
    (3.8916678022926607, -2.772484153217685, 0.5060648178503393),
    (3.285753657755655, -2.3681294933425376, 0.46449024233003106),
    (2.3465413258596377, -1.7097828382687081, 0.42323551169305323),
]


def truc_giao_hoa(G, ns_steps=5, ghi_log=False):
    X = G / (G.norm() * 1.01 + 1e-6)
    for i, (a, b, c) in enumerate(polar_express_coeffs[:ns_steps]):
        A = X @ X.mT if X.size(-2) <= X.size(-1) else X.mT @ X
        B = b * A + c * (A @ A)
        X = a * X + (B @ X if X.size(-2) <= X.size(-1) else X @ B)
        if ghi_log:
            sv = torch.linalg.svdvals(X)
            print(f"  sau vòng {i + 1}: giá trị kỳ dị min {sv.min():.4f} | max {sv.max():.4f} | trung bình {sv.mean():.4f}")
    return X


G = torch.randn(64, 128) @ torch.diag(torch.logspace(0, -2, 128))   # phổ rất lệch, như gradient thật
sv0 = torch.linalg.svdvals(G)
print(f"gradient ban đầu: giá trị kỳ dị min {sv0.min():.4f} | max {sv0.max():.4f} | tỷ lệ {sv0.max() / sv0.min():.0f}×\n")
X = truc_giao_hoa(G, ghi_log=True)

U, S, Vh = torch.linalg.svd(G, full_matrices=False)
chuan = U @ Vh
print(f"\nso với UVᵀ tính bằng SVD thật: sai khác lớn nhất {(X - chuan).abs().max():.4f}")
print("(5 vòng nhân ma trận thay cho một lần SVD — đó là toàn bộ mẹo của Muon.)")

Muon **không** dùng cho embedding và `lm_head`. Lý do: các ma trận đó có một chiều là vocab, mỗi hàng ứng với một
token riêng biệt, và trực giao hoá sẽ trộn các hàng đó với nhau theo cách không có ý nghĩa. Ngoài ra chúng thưa
theo bản chất — mỗi bước chỉ vài nghìn token xuất hiện — nên AdamW với learning rate riêng phù hợp hơn.

Trong `base_train.py`, bốn nhóm learning rate: `embedding_lr = 0.3`, `unembedding_lr = 0.008`,
`matrix_lr = 0.02` (Muon), `scalar_lr = 0.5`. Chênh nhau tới 37 lần, và đó là siêu tham số đã được nanochat dò
sẵn — project không đổi, chỉ ghim lại theo batch qua `--scaling-batch-size` (notebook 03 mục 8).

## 9. Ghép lại: một model nhỏ chạy được

Dưới đây là toàn bộ khối Transformer của nanochat trong ~40 dòng, giữ đúng thứ tự phép tính: RMSNorm trước mỗi
khối con, RoPE trên q/k, value embedding cộng vào V ở lớp có nó, residual cộng dồn.

In [ ]:
class KhoiNho(torch.nn.Module):
    def __init__(self, d, n_head, co_ve):
        super().__init__()
        self.n_head, self.dh, self.co_ve = n_head, d // n_head, co_ve
        self.c_q = torch.nn.Linear(d, d, bias=False)
        self.c_k = torch.nn.Linear(d, d, bias=False)
        self.c_v = torch.nn.Linear(d, d, bias=False)
        self.c_proj = torch.nn.Linear(d, d, bias=False)
        self.c_fc = torch.nn.Linear(d, 4 * d, bias=False)
        self.mlp_proj = torch.nn.Linear(4 * d, d, bias=False)
        self.ve_gate = torch.nn.Linear(12, n_head, bias=False) if co_ve else None   # 12 kênh, 1 cổng/đầu KV

    def forward(self, x, cos, sin, ve=None):
        B, T, d = x.shape
        h = rmsnorm(x)
        q = self.c_q(h).view(B, T, self.n_head, self.dh)
        k = self.c_k(h).view(B, T, self.n_head, self.dh)
        v = self.c_v(h).view(B, T, self.n_head, self.dh)
        if self.co_ve and ve is not None:                     # value embedding + cổng 3·sigmoid
            cong = 3 * torch.sigmoid(self.ve_gate(h[..., :12])).unsqueeze(-1)
            v = v + cong * ve.view(B, T, self.n_head, self.dh)
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        a = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True)
        x = x + self.c_proj(a.transpose(1, 2).reshape(B, T, d))
        x = x + self.mlp_proj(F.relu(self.c_fc(rmsnorm(x))) ** 2)     # ReLU²
        return x


class ModelNho(torch.nn.Module):
    def __init__(self, vocab, d=128, L=4, n_head=4):
        super().__init__()
        self.wte = torch.nn.Embedding(vocab, d)
        self.value_embeds = torch.nn.ModuleDict(
            {str(i): torch.nn.Embedding(vocab, d) for i in range(L) if has_ve(i, L)})
        self.blocks = torch.nn.ModuleList([KhoiNho(d, n_head, has_ve(i, L)) for i in range(L)])
        self.lm_head = torch.nn.Linear(d, vocab, bias=False)
        self.d, self.n_head = d, n_head

    def forward(self, idx):
        B, T = idx.shape
        cos, sin = rope_cos_sin(T, self.d // self.n_head)
        x = self.wte(idx)
        for i, blk in enumerate(self.blocks):
            ve = self.value_embeds[str(i)](idx) if str(i) in self.value_embeds else None
            x = blk(x, cos, sin, ve)
        return self.lm_head(rmsnorm(x))


torch.manual_seed(0)
m = ModelNho(vocab=500)
idx = torch.randint(0, 500, (2, 24))
logits = m(idx)
print(f"tham số: {sum(p.numel() for p in m.parameters()):,}")
print(f"đầu vào {tuple(idx.shape)} -> logits {tuple(logits.shape)}")
print(f"tổng xác suất tại một vị trí: {torch.softmax(logits[0, 5], -1).sum():.6f}")

### Kiểm tra tính chất mà `vitok/eval.py` dựa vào

Khi chấm điểm, project gom nhiều văn bản vào một batch và **đệm bên phải** cho bằng độ dài. Việc đó chỉ hợp lệ
nếu attention nhân quả bảo đảm: **token đệm ở phía sau không ảnh hưởng tới vị trí thật ở phía trước**.

Đây đúng là nội dung bài test `test_padding_does_not_change_nats` trong `tests/`. Cell dưới kiểm chứng lại trên
model vừa dựng — nếu tính chất này sai thì mọi con số bpc của project đều sai.

In [ ]:
x_ngan = torch.randint(0, 500, (1, 10))
x_dem = torch.cat([x_ngan, torch.zeros(1, 14, dtype=torch.long)], dim=1)      # đệm 14 token bên phải

with torch.no_grad():
    l1 = m(x_ngan)
    l2 = m(x_dem)[:, :10]
print(f"sai khác lớn nhất trên 10 vị trí thật: {(l1 - l2).abs().max():.3e}")
print("=> đệm bên phải không đổi kết quả, vì mask nhân quả chặn mọi vị trí sau.")

x_doi = x_dem.clone()
x_doi[0, 10:] = torch.randint(0, 500, (14,))                                  # đổi hẳn phần đệm
with torch.no_grad():
    l3 = m(x_doi)[:, :10]
print(f"đổi toàn bộ phần đệm: sai khác {(l1 - l3).abs().max():.3e}  (vẫn ở mức sai số dấu phẩy động)")

print("\nngược lại, đệm bên TRÁI thì hỏng:")
x_trai = torch.cat([torch.zeros(1, 14, dtype=torch.long), x_ngan], dim=1)
with torch.no_grad():
    l4 = m(x_trai)[:, 14:]
print(f"  sai khác: {(l1 - l4).abs().max():.3e}  <- khác 0: token thật giờ attention được tới 14 token đệm")
print("  đứng TRƯỚC chúng, mà mặt nạ nhân quả không chặn quá khứ.")

# Còn RoPE thì sao? Mọi token thật dịch đi cùng 14 vị trí, nên HIỆU vị trí giữa chúng không đổi.
# Kiểm chứng: đệm trái nhưng chặn luôn các token đệm khỏi attention -> kết quả phải trùng lại.
with torch.no_grad():
    cos, sin = rope_cos_sin(24, m.d // m.n_head)
    x = m.wte(x_trai)
    chan = torch.zeros(24, 24, dtype=torch.bool)
    chan[:, :14] = True                                   # không ai được nhìn token đệm
    chan |= torch.triu(torch.ones(24, 24, dtype=torch.bool), 1)
    for i, blk in enumerate(m.blocks):
        ve = m.value_embeds[str(i)](x_trai) if str(i) in m.value_embeds else None
        B_, T_, d_ = x.shape
        h = rmsnorm(x)
        q = blk.c_q(h).view(B_, T_, blk.n_head, blk.dh)
        k = blk.c_k(h).view(B_, T_, blk.n_head, blk.dh)
        v = blk.c_v(h).view(B_, T_, blk.n_head, blk.dh)
        if blk.co_ve:
            v = v + (3 * torch.sigmoid(blk.ve_gate(h[..., :12]))).unsqueeze(-1) * ve.view(B_, T_, blk.n_head, blk.dh)
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        mask = ~chan
        mask[:14, :14] = torch.eye(14, dtype=torch.bool)        # để hàng của token đệm không rỗng
        a = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), attn_mask=mask)
        x = x + blk.c_proj(a.transpose(1, 2).reshape(B_, T_, d_))
        x = x + blk.mlp_proj(F.relu(blk.c_fc(rmsnorm(x))) ** 2)
    l5 = m.lm_head(rmsnorm(x))[:, 14:]
print(f"\nđệm trái NHƯNG chặn token đệm khỏi attention: sai khác {(l1 - l5).abs().max():.3e}")
print("=> vị trí dịch đi 14 không làm gì cả: RoPE chỉ phụ thuộc HIỆU vị trí (mục 2).")

## 10. Tóm tắt

| Thành phần | Công thức chính | Vai trò trong project |
|---|---|---|
| Attention | $\mathrm{softmax}(QK^\top/\sqrt{d_h})V$ | bản thân nó không biết thứ tự |
| RoPE | quay góc $t\theta_i$, $\theta_i = \text{base}^{-2i/d_h}$ | vị trí tương đối; base = 100.000 |
| GQA | $h_{kv} \le h$, dùng chung K/V | project đặt $h_{kv} = h$ |
| KV cache | giữ $K, V$ quá khứ | sinh $O(T)$ thay vì $O(T^2)$; tốn $2LTh_{kv}d_h$ phần tử |
| Cửa sổ trượt | $O(Tw)$ thay $O(T^2)$ | tắt (`L`): trên T4 chỉ mô phỏng được bằng mặt nạ, không tiết kiệm gì |
| Value embedding | $V + 3\sigma(W_g\tilde x_{[:12]})\odot E^{(l)}[\text{tok}]$ | 42% tham số của d10 |
| RMSNorm | $x/\sqrt{\overline{x^2} + \epsilon}$, không có trọng số học | rẻ hơn LayerNorm |
| Softcap logits | $15\tanh(z/15)$ | chặn logits trong $(-15, 15)$ |
| ReLU² | $\mathrm{relu}(z)^2$ | thay GELU |
| Muon | thay bước cập nhật bằng $UV^\top$, xấp xỉ bằng Newton–Schulz 5 vòng | dùng cho ma trận thân, không dùng cho embedding |

## 11. Câu hỏi tự kiểm

1. Vì sao attention không mask là hàm đối xứng theo thứ tự token, và điều đó buộc ta phải làm gì?
2. Tính chất nào của phép quay làm RoPE mã hoá được vị trí **tương đối**?
3. Tăng `base` của RoPE từ 10.000 lên 100.000 giúp gì?
4. Vì sao KV cache hợp lệ — cái gì bảo đảm $K, V$ cũ không đổi khi có token mới?
5. Tính bộ nhớ KV cache của d10 ở ngữ cảnh 4.096 token, fp16.
6. GQA tiết kiệm bộ nhớ ở đâu, và vì sao nó không tiết kiệm nhiều tham số?
7. Vì sao Muon không dùng cho `lm_head`?
8. Nếu đệm **bên trái** thay vì bên phải khi chấm điểm, kết quả sai ở chỗ nào?

**Đáp án gợi ý**

1. Vì softmax và phép nhân ma trận không dùng chỉ số vị trí; phải thêm thông tin vị trí bằng embedding vị trí
   hoặc RoPE.
2. Phép quay bảo toàn tích vô hướng và hợp thành cộng góc, nên $\langle R_mq, R_nk\rangle = \langle R_{m-n}q, k\rangle$.
3. Làm bước sóng của các cặp chiều chậm dài ra, nên model phân biệt được vị trí ở ngữ cảnh dài hơn.
4. Attention nhân quả: $K_j, V_j$ chỉ phụ thuộc token $\le j$, nên token mới ở vị trí sau không làm chúng đổi.
5. $2 \times 10 \times 4096 \times 5 \times 128 \times 2$ byte $\approx 105$ MB.
6. Ở KV cache (tỷ lệ $h_{kv}$); tham số chỉ giảm ở hai ma trận `c_k`, `c_v`, phần nhỏ so với toàn model.
7. Vì mỗi hàng của nó ứng với một token riêng; trực giao hoá sẽ trộn các hàng đó, và gradient của nó vốn thưa.
8. Token thật attention được tới các token đệm đứng trước chúng (mặt nạ nhân quả không chặn quá khứ), nên logits
   đổi. Việc vị trí tuyệt đối dịch đi thì **không** gây sai, vì RoPE chỉ phụ thuộc hiệu vị trí — cell cuối mục 9
   kiểm chứng cả hai điều.

**Nguồn đọc thêm**

- [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864) — RoPE gốc.
- [GQA: Training Generalized Multi-Query Transformer Models](https://arxiv.org/abs/2305.13245).
- [Muon: An optimizer for hidden layers](https://kellerjordan.github.io/posts/muon/) — trang gốc của Muon.
- [The Polar Express](https://arxiv.org/abs/2505.16932) — bộ hệ số lặp nanochat đang dùng.
- `third_party/nanochat/nanochat/gpt.py` — bản cài đặt mà project chạy.